# H2-1 확장 — 3단계: 가구×주차 패널 구성 (33~101주)

**이전 단계 반영**: 계층 고정 구간을 17~32주로 정정(팀 안정구간 원칙 17주 이후에 맞춤).

**이번 단계에서 하는 일**
1. (household_key, WEEK_NO) 골격에 주간 지출액을 붙임
2. TypeA/B/C 캠페인 활성 여부를 가구×주차 단위로 붙임 (상호배타적이지 않은 개별 더미)
3. 계층 라벨 부착
4. **지난 해석에서 남겨둔 질문 — "2개 이상 타입을 같은 주에 동시에 받았는가?"를 확인**
5. 본 회귀 전 참고용 예비 비교 (고정효과 통제 전 단순 평균)


In [3]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)

DATA_DIR = "data/"

TIER_MIN_WEEK = 17   # 정정: 팀 안정구간 기준(17주 이후)
TIER_MAX_WEEK = 32
CAMP_MIN_WEEK = 33
CAMP_MAX_WEEK = 101


## (재구성) 1~2단계 — 17주 기준으로 다시 계산

이전 산출물을 그대로 이어받기 위해 1~2단계를 17주 기준으로 다시 실행합니다.

In [4]:
tx = pd.read_csv(DATA_DIR + "transaction_data.csv",
                  usecols=["household_key", "DAY", "WEEK_NO", "SALES_VALUE"])
campaign_table = pd.read_csv(DATA_DIR + "campaign_table.csv")
campaign_desc  = pd.read_csv(DATA_DIR + "campaign_desc.csv")

tier_window = tx[(tx["WEEK_NO"] >= TIER_MIN_WEEK) & (tx["WEEK_NO"] <= TIER_MAX_WEEK)]
n_weeks_tier = TIER_MAX_WEEK - TIER_MIN_WEEK + 1
avg_weekly_spend = (tier_window.groupby("household_key")["SALES_VALUE"].sum() / n_weeks_tier).rename("avg_weekly_spend_pre")
all_households = tx["household_key"].unique()
avg_weekly_spend = avg_weekly_spend.reindex(all_households).fillna(0)

tier5 = pd.qcut(avg_weekly_spend, 5, labels=["1분위(최저)", "2분위", "3분위", "4분위", "5분위(최고)"])
tier3 = pd.qcut(avg_weekly_spend, 3, labels=["저지출", "중지출", "고지출"])
tier_df = pd.DataFrame({"avg_weekly_spend_pre": avg_weekly_spend, "tier5": tier5, "tier3": tier3})
tier_df["was_zero_pre"] = (tier_df["avg_weekly_spend_pre"] == 0)

print(f"[재확인] 계층 고정 구간: {TIER_MIN_WEEK}~{TIER_MAX_WEEK}주 ({n_weeks_tier}주)")
print(f"사전기간 거래 0인 가구(바닥효과 대상): {tier_df['was_zero_pre'].sum()}")
print(f"3계층 분포: {tier_df['tier3'].value_counts().to_dict()}")


[재확인] 계층 고정 구간: 17~32주 (16주)
사전기간 거래 0인 가구(바닥효과 대상): 142
3계층 분포: {'저지출': 834, '중지출': 834, '고지출': 832}


In [5]:
day_to_week = tx[["DAY", "WEEK_NO"]].drop_duplicates().sort_values("DAY").reset_index(drop=True)
day_arr = day_to_week["DAY"].values
week_arr = day_to_week["WEEK_NO"].values

def day_to_week_lookup(day):
    idx = np.searchsorted(day_arr, day, side="right") - 1
    idx = max(0, min(idx, len(day_arr) - 1))
    return week_arr[idx]

campaign_desc = campaign_desc.copy()
campaign_desc["START_WEEK"] = campaign_desc["START_DAY"].apply(day_to_week_lookup)
campaign_desc["END_WEEK"] = campaign_desc["END_DAY"].apply(day_to_week_lookup)

camp_full = campaign_table.merge(
    campaign_desc[["CAMPAIGN", "DESCRIPTION", "START_WEEK", "END_WEEK"]],
    on="CAMPAIGN", how="left", suffixes=("", "_desc")
)
recipients = set(campaign_table["household_key"].unique())
never_recipients = set(all_households) - recipients
print("1~2단계 재구성 완료")


1~2단계 재구성 완료


## 3단계 — 가구×주차 패널 구성 (33~101주)

### 3-1. 기본 패널 골격 + 주간 지출액

In [6]:
weeks_camp = list(range(CAMP_MIN_WEEK, CAMP_MAX_WEEK + 1))
n_weeks_camp = len(weeks_camp)
print(f"캠페인 관찰 구간: {CAMP_MIN_WEEK}~{CAMP_MAX_WEEK}주 ({n_weeks_camp}주)")
print(f"패널 예상 행수: {len(all_households):,} 가구 x {n_weeks_camp}주 = {len(all_households)*n_weeks_camp:,}")

panel_index = pd.MultiIndex.from_product([all_households, weeks_camp], names=["household_key", "WEEK_NO"])
panel = pd.DataFrame(index=panel_index).reset_index()

weekly_spend = (
    tx[(tx["WEEK_NO"] >= CAMP_MIN_WEEK) & (tx["WEEK_NO"] <= CAMP_MAX_WEEK)]
    .groupby(["household_key", "WEEK_NO"])["SALES_VALUE"].sum()
    .rename("spend")
)
panel = panel.merge(weekly_spend, on=["household_key", "WEEK_NO"], how="left")
panel["spend"] = panel["spend"].fillna(0)

print(f"\n실제 생성된 패널 행수: {len(panel):,}")
print(f"주당 지출 0인 행 비율: {(panel['spend']==0).mean()*100:.1f}% (그 주에 아예 안 온 가구주차 조합)")


캠페인 관찰 구간: 33~101주 (69주)
패널 예상 행수: 2,500 가구 x 69주 = 172,500

실제 생성된 패널 행수: 172,500
주당 지출 0인 행 비율: 46.9% (그 주에 아예 안 온 가구주차 조합)


### 3-2. 캠페인 타입별 활성 플래그 (벡터화 처리)

In [7]:
def build_active_weeks(camp_df, type_name):
    sub = camp_df[camp_df["DESCRIPTION_desc"] == type_name][["household_key", "START_WEEK", "END_WEEK"]].copy()
    sub["START_WEEK"] = sub["START_WEEK"].clip(lower=CAMP_MIN_WEEK)
    sub["END_WEEK"] = sub["END_WEEK"].clip(upper=CAMP_MAX_WEEK)
    sub["WEEK_NO"] = sub.apply(lambda r: list(range(int(r["START_WEEK"]), int(r["END_WEEK"]) + 1)), axis=1)
    sub = sub.explode("WEEK_NO")[["household_key", "WEEK_NO"]].drop_duplicates()
    sub["WEEK_NO"] = sub["WEEK_NO"].astype(int)
    sub[f"active_{type_name}"] = 1
    return sub

for t in ["TypeA", "TypeB", "TypeC"]:
    active = build_active_weeks(camp_full, t)
    panel = panel.merge(active, on=["household_key", "WEEK_NO"], how="left")
    panel[f"active_{t}"] = panel[f"active_{t}"].fillna(0).astype(int)
    n_active_pairs = int(panel[f"active_{t}"].sum())
    n_active_hh = panel.loc[panel[f"active_{t}"]==1, "household_key"].nunique()
    print(f"{t}: 활성 가구x주차 조합 {n_active_pairs:,}건 / 관여 가구 {n_active_hh:,}명")


TypeA: 활성 가구x주차 조합 28,654건 / 관여 가구 1,513명
TypeB: 활성 가구x주차 조합 13,260건 / 관여 가구 1,023명
TypeC: 활성 가구x주차 조합 5,788건 / 관여 가구 397명


### 3-3. 계층 라벨 부착 + 패널 확인

In [8]:
panel = panel.merge(tier_df[["tier3", "tier5", "was_zero_pre"]], left_on="household_key", right_index=True, how="left")

print(f"최종 패널 shape: {panel.shape}")
print()
print(panel.head(8).to_string(index=False))


최종 패널 shape: (172500, 9)

 household_key  WEEK_NO  spend  active_TypeA  active_TypeB  active_TypeC tier3   tier5  was_zero_pre
          2375       33   0.00             0             0             0   저지출 1분위(최저)          True
          2375       34   0.00             0             0             0   저지출 1분위(최저)          True
          2375       35  15.81             0             0             0   저지출 1분위(최저)          True
          2375       36   0.00             0             0             0   저지출 1분위(최저)          True
          2375       37  39.95             0             0             0   저지출 1분위(최저)          True
          2375       38   0.00             0             0             0   저지출 1분위(최저)          True
          2375       39   0.00             0             0             0   저지출 1분위(최저)          True
          2375       40   0.00             0             0             0   저지출 1분위(최저)          True


### 3-4. [핵심 검증] 같은 주에 2개 이상 타입을 동시에 받은 경우가 있는가?

지난 해석에서 "1,008개 가구가 2개 이상 타입을 받았는데, 동시인지 시간차인지 확인 필요"라고 남겼던 부분.

In [9]:
panel["n_types_active_this_week"] = panel[["active_TypeA", "active_TypeB", "active_TypeC"]].sum(axis=1)
same_week_overlap = panel[panel["n_types_active_this_week"] >= 2]
print(f"같은 주에 2개 이상 타입이 동시 활성인 가구x주차 조합 수: {len(same_week_overlap):,}")
print(f"관련 고유 가구 수: {same_week_overlap['household_key'].nunique():,}")

combo = same_week_overlap.apply(
    lambda r: "+".join([t for t in ["TypeA","TypeB","TypeC"] if r[f"active_{t}"]==1]), axis=1
)
print()
print("[동시 활성 조합별 건수]")
print(combo.value_counts())

n_multi_type_total = camp_full.groupby("household_key")["DESCRIPTION_desc"].nunique().gt(1).sum()
n_same_week_hh = same_week_overlap['household_key'].nunique()
print(f"\n2개 이상 타입 수신 가구(시간차 포함, 총): {n_multi_type_total:,}")
print(f"그중 실제로 '같은 주'에 겹친 가구: {n_same_week_hh:,} ({n_same_week_hh/n_multi_type_total*100:.1f}%)")


같은 주에 2개 이상 타입이 동시 활성인 가구x주차 조합 수: 7,110
관련 고유 가구 수: 869

[동시 활성 조합별 건수]
TypeA+TypeB          3530
TypeA+TypeC          1585
TypeB+TypeC          1434
TypeA+TypeB+TypeC     561
Name: count, dtype: int64

2개 이상 타입 수신 가구(시간차 포함, 총): 1,008
그중 실제로 '같은 주'에 겹친 가구: 869 (86.2%)


### 3-5. 예비 확인 — 캠페인 활성 주 vs 비활성 주 평균 지출 (고정효과 통제 전, 참고용)

In [10]:
panel["any_active"] = (panel["n_types_active_this_week"] >= 1).astype(int)
prelim = panel.groupby(["tier3", "any_active"], observed=True)["spend"].mean().unstack()
prelim.columns = ["캠페인 비활성주 평균지출", "캠페인 활성주 평균지출"]
prelim["차이"] = prelim["캠페인 활성주 평균지출"] - prelim["캠페인 비활성주 평균지출"]
prelim["변화율(%)"] = (prelim["차이"] / prelim["캠페인 비활성주 평균지출"] * 100)
print(prelim.round(2))
print()
print("※ 가구·주차 고정효과를 통제하지 않은 단순 평균 비교. 6단계 본 회귀 결과와 다를 수 있음")


       캠페인 비활성주 평균지출  캠페인 활성주 평균지출     차이  변화율(%)
tier3                                            
저지출            11.54         39.49  27.95  242.27
중지출            22.48         36.08  13.60   60.50
고지출            61.68         75.61  13.93   22.58

※ 가구·주차 고정효과를 통제하지 않은 단순 평균 비교. 6단계 본 회귀 결과와 다를 수 있음


**참고용 관찰**: 저지출 계층의 변화율(+242%)이 중지출(+60%)·고지출(+23%)보다 훨씬 커 보이는데,
이 시점에서 두 가지 경쟁 해석이 가능하고 **아직 구분할 수 없습니다**.

1. 저지출 고객이 캠페인에 실제로 더 강하게 반응한다
2. 지난 해석에서 지적한 **바닥효과**(사전기간 거래 0인 142개 가구가 저지출 계층에 몰려 있어,
   조금만 지출해도 %로는 폭증하는 것처럼 보임) + 가구·주차 고정효과를 아직 통제하지 않은 단순평균의 착시

→ 이 표는 절대 결론에 쓰지 않고, 6단계 본 회귀(고정효과 통제) 결과와 반드시 대조해서만 해석합니다.

## 다음 단계
3단계에서 다중공선성 이슈가 새로 발견되었으므로, 4단계(Type×계층 셀 크기표)로 넘어가기 전에
**이 동시노출 문제를 어떻게 처리할지 먼저 결정**하는 게 좋겠습니다.

In [11]:
!pip install statsmodels --break-system-packages -q   # 필요시 최초 1회만

import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ============================================================
# VIF 진단 1 — 원본 Type 활성 더미 3개 (가장 기본적인 확인)
# ============================================================
X_raw = panel[["active_TypeA", "active_TypeB", "active_TypeC"]].astype(float)

vif_raw = pd.DataFrame({
    "변수": X_raw.columns,
    "VIF": [variance_inflation_factor(X_raw.values, i) for i in range(X_raw.shape[1])]
})
print("[VIF 진단 1] 원본 Type 활성 더미 (raw)")
print(vif_raw)
print()

# ============================================================
# VIF 진단 2 — 실제 6단계 회귀에 들어갈 Type×계층 상호작용 더미 9개
#            (진짜 중요한 건 이쪽 — 원본 더미가 아니라 상호작용 항끼리 공선성이 문제)
# ============================================================
interaction_cols = []
for t in ["TypeA", "TypeB", "TypeC"]:
    for tier in ["저지출", "중지출", "고지출"]:
        col = f"{t}_x_{tier}"
        panel[col] = ((panel[f"active_{t}"] == 1) & (panel["tier3"] == tier)).astype(int)
        interaction_cols.append(col)

X_inter = panel[interaction_cols].astype(float)

vif_inter = pd.DataFrame({
    "변수": X_inter.columns,
    "VIF": [variance_inflation_factor(X_inter.values, i) for i in range(X_inter.shape[1])]
})
print("[VIF 진단 2] Type×계층 상호작용 더미 (본 회귀 투입 변수)")
print(vif_inter.sort_values("VIF", ascending=False))
print()

# ============================================================
# 참고 — 원본 Type 더미 간 상관계수 (VIF와 같이 보면 해석에 도움)
# ============================================================
corr_raw = X_raw.corr()
print("[참고] Type 활성 더미 간 상관계수")
print(corr_raw.round(3))

# ============================================================
# 판단 기준 (참고용 각주)
# ============================================================
print("""
판단 기준(경험칙):
  VIF < 5   : 문제없음, 그대로 진행
  5 <= VIF < 10 : 주의, 결과 해석 시 해당 변수의 표준오차가 넓다는 점 명시
  VIF >= 10 : 심각, (b)동시노출 제외 또는 (a)조합별 더미 등 조치 필요
""")

ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


[VIF 진단 1] 원본 Type 활성 더미 (raw)
             변수       VIF
0  active_TypeA  1.062637
1  active_TypeB  1.089636
2  active_TypeC  1.071389

[VIF 진단 2] Type×계층 상호작용 더미 (본 회귀 투입 변수)
            변수       VIF
5  TypeB_x_고지출  1.117451
8  TypeC_x_고지출  1.098625
2  TypeA_x_고지출  1.079771
0  TypeA_x_저지출  1.044315
4  TypeB_x_중지출  1.039543
3  TypeB_x_저지출  1.037779
1  TypeA_x_중지출  1.036125
7  TypeC_x_중지출  1.023822
6  TypeC_x_저지출  1.016183

[참고] Type 활성 더미 간 상관계수
              active_TypeA  active_TypeB  active_TypeC
active_TypeA         1.000         0.110         0.102
active_TypeB         0.110         1.000         0.187
active_TypeC         0.102         0.187         1.000

판단 기준(경험칙):
  VIF < 5   : 문제없음, 그대로 진행
  5 <= VIF < 10 : 주의, 결과 해석 시 해당 변수의 표준오차가 넓다는 점 명시
  VIF >= 10 : 심각, (b)동시노출 제외 또는 (a)조합별 더미 등 조치 필요

